# KOA Onboarding

Este notebook lê arquivos TXT de contratos (gerados a partir de seus PDFs) e gera modelos Prolog (um para cada contrato) usando chamada ao Gemini.
A ideia nesta primeira abordagem é gerar os modelos que serão utilizados na construção/extensão da ontologia de domínio.


Entradas: contratos em .txt no diretório configurado.

Saídas: arquivos KOA_<contrato>.pl no diretório de persistência.

In [1]:
# Monta o Google Drive no ambiente Colab para acessar os arquivos de contratos e salvar os .pl gerados
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
# Importa bibliotecas padrão e do Gemini
from pathlib import Path
import os
import re
import importlib.util, types

import google.generativeai as genai
from google.generativeai.types import GenerationConfig

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
# Recupera chave Gemini com secrets do Colab
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [4]:
# Configura pastas para entradas e saídas
directory = '/content/drive/My Drive/KOA/contratos/'
persist_directory = '/content/drive/My Drive/KOA/onboarding/modelos'

In [5]:
# Monta lista de arquivos que serão processados pelo onboarding
files_txt = {
    '195_2022_Brasoftware.txt',
}

In [6]:
# Define função para leitura de arquivos .txt
def read_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return file.read()

# Geração do Prolog

Define a classe responsável por:

- Ler o texto do contrato (_read_document)

- Montar um prompt para extrair fatos do texto e estruturar um modelo Prolog (modelo base) previamente estruturado

- Chamar o Gemini (genai.GenerativeModel(...).generate_content(...))

- Retornar o Prolog gerado (generate_prolog_ontology)

Entradas: caminho do contrato .txt

Saídas: string com Prolog base (modelo base)

Pendências:
1) Truncamento no retorno da chamada ao Gemini, por conta do tamanho da janela de saída;
2) Modelo organizado por cláusulas feito aqui é o melhor para este caso de uso?


In [7]:
# Funções de apoio

CONTINUE_MARKER = "%% CONTINUE"

def _clean_prolog_text(x: str) -> str:
    """Remove cercas de markdown caso o modelo insista em colocar."""
    if not x:
        return ""
    x = re.sub(r"```(?:prolog|pl|)\s*", "", x, flags=re.I)
    x = x.replace("```", "")
    return x.strip()

def _strip_continue_marker(x: str) -> tuple[str, bool]:
    """
    Remove o marcador %% CONTINUE do texto e indica se ele existia.
    Aceita variações com espaços, e garante remoção mesmo se vier no fim sem newline.
    """
    if not x:
        return "", False

    hit = CONTINUE_MARKER in x
    # remove todas as ocorrências por segurança
    x = x.replace(CONTINUE_MARKER, "")
    return x, hit

def _tail_lines(s: str, n: int = 80) -> str:
    lines = s.splitlines()
    return "\n".join(lines[-n:]) if len(lines) > n else s

def _merge_by_overlap(prev: str, nxt: str, max_overlap_lines: int = 250) -> str:
    """Merge tentando evitar duplicação por overlap de linhas."""
    a = _clean_prolog_text(prev)
    b = _clean_prolog_text(nxt)

    if not a:
        return (b.strip() + "\n") if b else ""
    if not b:
        return (a.strip() + "\n") if a else ""

    a_lines = a.splitlines()
    b_lines = b.splitlines()

    max_k = min(len(a_lines), len(b_lines), max_overlap_lines)
    overlap = 0
    for k in range(max_k, 0, -1):
        if a_lines[-k:] == b_lines[:k]:
            overlap = k
            break

    merged = a_lines + b_lines[overlap:]
    return "\n".join(merged).strip() + "\n"

def _get_finish_reason(resp) -> str:
    try:
        fr = resp.candidates[0].finish_reason
        return fr.name if hasattr(fr, "name") else str(fr)
    except Exception:
        return ""

In [8]:
# Classe principal para chamada ao Gemini e geração do Prolog
class OntologyGenerationAgent:
    """
    Lê arquivo TXT e chama o Gemini para gerar um modelo Prolog.
    """
    def __init__(self, file_path: str):
        self.file_path = file_path
        self.content = self._read_document()
        self.prolog_ontology = ""

        # Configure the Gemini API with the environment variable
        try:
            gemini_api_key = os.environ.get("GOOGLE_API_KEY")
            if not gemini_api_key:
                raise ValueError("Please set the GOOGLE_API_KEY environment variable.")
            genai.configure(api_key=gemini_api_key)
        except Exception as e:
            print(f"Error configuring Gemini API: {e}")
            raise

    def _read_document(self) -> str:
        try:
            with open(self.file_path, 'r', encoding='utf-8') as f:
                return f.read()
        except FileNotFoundError:
            print(f"Error: The file '{self.file_path}' was not found.")
            return ""
        except Exception as e:
            print(f"Error reading the file: {e}")
            return ""

    def _generate_ontology_with_gemini(self) -> str:
        """
        Prompts SEMPRE em inglês!!!
        """

        prompt_base = f"""
        You are a Knowledge Representation expert specializing in Legal Ontologies and Prolog.

        Your task is to perform a STRICT, STANDARDIZED, and DOCUMENT-CENTRIC onboarding of a contract into Prolog.

        The goal is NOT to interpret, infer, or reason about the contract, but to ORGANIZE its content into a normalized factual structure that will later support semantic reasoning.

        ==================================================
        GLOBAL CONSTRAINTS
        ==================================================

        1. Language:
          - Use Portuguese for all strings, identifiers, atom values, and labels.
          - Predicate names MUST be in English, as specified below.
          - Normalize all identifiers using lowercase, underscores, and ASCII characters only.

        2. Output Format:
          - Return ONLY valid Prolog code.
          - Do NOT include markdown, explanations, comments, or prose.
          - Every predicate MUST end with a period (.).

        3. No Inference:
          - Do NOT infer obligations, rights, risks, or meanings.
          - Do NOT create rules.
          - Do NOT merge or reinterpret clauses.
          - ONLY extract and organize explicit information from the text.

        ==================================================
        MANDATORY GLOBAL CONTRACT METADATA
        ==================================================

        1. Define exactly ONE contract identifier:
          - contract(contract_id).

        2. Extract all identification and header information appearing before the clauses, such as:
          - contract number
          - SAP / OCS numbers
          - parties
          - dates
          - object
          - global values

        3. Represent metadata ONLY using:
          - contract_metadata(contract_id, key, value).

        4. Metadata facts MUST NOT reference any clause.

        ==================================================
        MANDATORY CLAUSE STRUCTURE
        ==================================================

        1. Every contractual clause MUST be represented using:
          - contract_clause(contract_id, clause_id, clause_title, clause_text).

        2. Clause identifiers:
          - MUST be derived from the clause titles.
          - MUST be normalized (lowercase, underscores, ASCII).
          - Example:
            "Cláusula Terceira – Prazo de Vigência"
            -> clausula_terceira_prazo_vigencia

        3. Clause text:
          - MUST contain the full literal text of the clause.
          - MUST be enclosed in single quotes.

        ==================================================
        MANDATORY CLAUSE FACT EXTRACTION
        ==================================================

        1. From each clause, extract ATOMIC factual statements.

        2. Each atomic statement MUST be represented as:
          - contract_clause_fact(
              contract_id,
              clause_id,
              fact_type,
              fact_data,
              evidence_ref
            ).

        3. Definitions:
          - fact_type:
              A short, normalized atom describing the type of information
              (e.g., payment_term, penalty, prazo, indice_reajuste, valor, condicao).
          - fact_data:
              A structured Prolog term or list containing ONLY the explicit data stated.
              Do NOT include interpretation or explanation.
          - evidence_ref:
              A short reference to the clause text (e.g., trecho_literal or span_placeholder).

        4. Each contract_clause_fact MUST:
          - Refer to exactly ONE clause.
          - Represent exactly ONE factual assertion.
          - Be short, structured, and repeatable across contracts.

        ==================================================
        ANNEXES (WHEN PRESENT)
        ==================================================

        1. If the contract contains annexes, represent each one as:
          - contract_annex(contract_id, annex_id, annex_title, annex_text).

        2. Extract facts from annexes using:
          - contract_annex_fact(
              contract_id,
              annex_id,
              fact_type,
              fact_data,
              evidence_ref
            ).

        ==================================================
        COMPLETENESS AND LIMIT HANDLING
        ==================================================

        1. Do NOT omit any clause or annex present in the text.

        2. If you cannot finish generating the full Prolog model due to output limits,
          end the response with EXACTLY this single line:
          %% CONTINUE

        3. If the model is complete, do NOT output %% CONTINUE.

        Text to analyze:
        ---
        {self.content}
        ---

        Prolog Model:
        """

        try:
            model = genai.GenerativeModel("gemini-2.0-flash-lite")
            gen_cfg = GenerationConfig(
                temperature=0.2,
                max_output_tokens=8192,
            )

            full = ""
            prompt = prompt_base

            for _ in range(6):  # limitando chamadas para não loopar
                resp = model.generate_content(prompt, generation_config=gen_cfg)
                part = resp.text or ""

                # 1) detecta marker
                part_no_marker, has_marker = _strip_continue_marker(part)

                # 2) detecta truncamento real pela API
                finish_reason = _get_finish_reason(resp)
                hit_max_tokens = ("MAX_TOKENS" in finish_reason)

                # decisão
                need_continue = has_marker or hit_max_tokens

                # merge
                full = _merge_by_overlap(full, part_no_marker)

                # log sem “False” colado no texto
                print("\n--- Gemini finish_reason:", finish_reason, "| need_continue:", need_continue, "---\n")

                if not need_continue:
                    break

                tail = _tail_lines(_clean_prolog_text(full), n=80)
                prompt = (
                    "Continue exactly from where you stopped.\n"
                    "Rules:\n"
                    "- Do NOT repeat or rewrite any previously generated content.\n"
                    "- Continue the Prolog code from the last incomplete predicate or term.\n"
                    "- Return ONLY valid Prolog code.\n"
                    "- Do NOT use markdown, code fences, comments, or explanations.\n"
                    f"- If you reach the output limit again, end with the single line: {CONTINUE_MARKER}\n\n"
                    "Last generated lines (for reference only):\n"
                    f"{tail}\n\n"
                    "Now continue:"
                )

            return _clean_prolog_text(full).rstrip() + "\n"

        except Exception as e:
            print(f"An error occurred while calling the Gemini API: {e}")
            return ""

    def generate_prolog_ontology(self):
        """Public method to generate the ontology."""
        self.prolog_ontology = self._generate_ontology_with_gemini()
        return self.prolog_ontology

In [9]:
if __name__ == "__main__":

    for file_to_process in sorted(files_txt):
        print("\n====================================================")
        print(f"Processing document: {file_to_process}")

        file_path = os.path.join(directory, file_to_process)
        if not os.path.exists(file_path):
            print(f"⚠️  Skipping (file not found): {file_path}")
            continue

        _doc_text = read_file(file_path)

        stem = Path(file_to_process).stem.replace(" ", "_")

        print("Generating base ontology...")
        ontology_generator = OntologyGenerationAgent(file_path=file_path)
        base_ontology = ontology_generator.generate_prolog_ontology()

        base_out = os.path.join(persist_directory, f"KOA_{stem}.pl")
        with open(base_out, "w", encoding="utf-8") as f:
            f.write(base_ontology)
        print(f"Base ontology saved to '{base_out}'")


Processing document: 195_2022_Brasoftware.txt
Generating base ontology...

--- Gemini finish_reason: MAX_TOKENS | need_continue: True ---


--- Gemini finish_reason: STOP | need_continue: True ---


--- Gemini finish_reason: STOP | need_continue: True ---


--- Gemini finish_reason: STOP | need_continue: True ---


--- Gemini finish_reason: STOP | need_continue: True ---


--- Gemini finish_reason: STOP | need_continue: True ---

Base ontology saved to '/content/drive/My Drive/KOA/onboarding/modelos/KOA_195_2022_Brasoftware.pl'
